# 🎯 Finalize & Balance Training Data v7

**Goal**: Create balanced dataset for XGBoost (50/50 Action vs Zero)  
**Problem**: 98% of rows have bolus=0 (class imbalance)  
**Solution**: Downsample zeros to match action rows

In [ ]:
!pip install polars -U -q

import polars as pl
import numpy as np
import os

try:
    from google.colab import drive
    drive.mount('/content/drive')
    BASE_PATH = "/content/drive/MyDrive/DiabetesKDS/data"
except:
    BASE_PATH = r"D:\Diabete\data"

print(f"✅ Polars: {pl.__version__}")

In [ ]:
# ============================================================
# CONFIGURATION
# ============================================================
INPUT_FILE = f"{BASE_PATH}/processed/master_train_v7_clean_core.parquet"
OUTPUT_FILE = f"{BASE_PATH}/processed/train_ready_balanced_v7.parquet"

# Safety thresholds
THRESHOLDS = {
    'CGM_MIN': 39, 'CGM_MAX': 400,
    'IOB_MAX': 30, 'BOLUS_MAX': 25, 'CARBS_MAX': 250
}

# Balance ratio (1.0 = equal, 1.5 = 50% more zeros)
ZERO_RATIO = 1.5

print(f"📁 Input:  {INPUT_FILE}")
print(f"📁 Output: {OUTPUT_FILE}")

## Step 1: Load & Filter (Sanitization)

In [ ]:
print("="*70)
print("📂 STEP 1: LOAD & SAFETY FILTERS")
print("="*70)

df = pl.read_parquet(INPUT_FILE)
original = len(df)
print(f"\n✅ Loaded: {original:,} rows")

# Pre-filter class distribution
action_pre = df.filter(pl.col('bolus') > 0).height
zero_pre = df.filter(pl.col('bolus') == 0).height
print(f"\n📊 Pre-Filter Distribution:")
print(f"   Action (bolus > 0): {action_pre:,} ({action_pre/original*100:.2f}%)")
print(f"   Zeros (bolus = 0):  {zero_pre:,} ({zero_pre/original*100:.2f}%)")

In [ ]:
# Apply safety filters
print("\n🧹 Applying Safety Filters...")

# CGM Range
df = df.filter(
    (pl.col('CGM') >= THRESHOLDS['CGM_MIN']) &
    (pl.col('CGM') <= THRESHOLDS['CGM_MAX'])
)
print(f"   After CGM filter: {len(df):,}")

# IOB Max
if 'iob' in df.columns:
    df = df.filter(
        (pl.col('iob') < THRESHOLDS['IOB_MAX']) | pl.col('iob').is_null()
    )
    print(f"   After IOB filter: {len(df):,}")

# Bolus Max
df = df.filter(
    (pl.col('bolus') < THRESHOLDS['BOLUS_MAX']) | pl.col('bolus').is_null()
)
print(f"   After Bolus filter: {len(df):,}")

# Carbs Max
df = df.filter(
    (pl.col('carbs') < THRESHOLDS['CARBS_MAX']) | pl.col('carbs').is_null()
)
print(f"   After Carbs filter: {len(df):,}")

# Drop warm-up nulls
for col in ['est_isf', 'est_icr']:
    if col in df.columns:
        df = df.filter(pl.col(col).is_not_null())
print(f"   After null filter: {len(df):,}")

filtered = len(df)
print(f"\n✅ After all filters: {filtered:,} rows ({filtered/original*100:.1f}% retained)")

## Step 2: The Great Balancing Act

In [ ]:
print("="*70)
print("⚖️ STEP 2: BALANCING DATASET")
print("="*70)

# Split into Action and Zeros
df_action = df.filter(pl.col('bolus') > 0)
df_zeros = df.filter(pl.col('bolus') == 0)

n_action = len(df_action)
n_zeros = len(df_zeros)

print(f"\n📊 Split Results:")
print(f"   🎯 Action rows (bolus > 0): {n_action:,}")
print(f"   ⚪ Zero rows (bolus = 0):   {n_zeros:,}")
print(f"   📈 Imbalance ratio: 1:{n_zeros/n_action:.1f}")

In [ ]:
# Downsample zeros
sample_size = int(n_action * ZERO_RATIO)
print(f"\n🎲 Downsampling zeros...")
print(f"   Target sample size: {sample_size:,} (ratio: {ZERO_RATIO}x)")

# Random sample
df_zeros_sampled = df_zeros.sample(n=min(sample_size, n_zeros), seed=42)
print(f"   Sampled: {len(df_zeros_sampled):,} zero rows")

# Merge
print("\n🔗 Merging datasets...")
df_balanced = pl.concat([df_action, df_zeros_sampled])
print(f"   Combined: {len(df_balanced):,} rows")

# Shuffle
print("\n🔀 Shuffling...")
df_balanced = df_balanced.sample(fraction=1.0, shuffle=True, seed=42)
print("   Shuffled ✅")

In [ ]:
# Final distribution
final_action = df_balanced.filter(pl.col('bolus') > 0).height
final_zeros = df_balanced.filter(pl.col('bolus') == 0).height
total = len(df_balanced)

print("\n" + "="*70)
print("📊 FINAL BALANCED DISTRIBUTION")
print("="*70)
print(f"\n   🎯 Action (bolus > 0): {final_action:,} ({final_action/total*100:.1f}%)")
print(f"   ⚪ Zeros (bolus = 0):  {final_zeros:,} ({final_zeros/total*100:.1f}%)")
print(f"   📊 Total:              {total:,}")
print(f"   ⚖️ Balance ratio:      1:{final_zeros/final_action:.2f}")

## Step 3: Save Output

In [ ]:
print("="*70)
print("💾 STEP 3: SAVING OUTPUT")
print("="*70)

os.makedirs(os.path.dirname(OUTPUT_FILE), exist_ok=True)
df_balanced.write_parquet(OUTPUT_FILE, compression='zstd')

if os.path.exists(OUTPUT_FILE):
    size_gb = os.path.getsize(OUTPUT_FILE) / (1024**3)
    print(f"\n✅ Saved: {OUTPUT_FILE}")
    print(f"   Size: {size_gb:.2f} GB")
    print(f"   Rows: {len(df_balanced):,}")

# Summary
print("\n" + "="*70)
print("📋 SUMMARY")
print("="*70)
print(f"   Original:      {original:,} rows")
print(f"   After filters: {filtered:,} rows")
print(f"   After balance: {total:,} rows")
print(f"   Action/Zero:   {final_action:,} / {final_zeros:,}")
print("\n🎉 BALANCED DATASET READY FOR XGBOOST!")

In [ ]:
# Verification
print("\n📋 Sample Data:")
df_balanced.head(10)